In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col, count, when, isnan
from pyspark.sql.window import Window
catalog_name = 'automobilerepair'

In [0]:
df = spark.read.table("automobilerepair.bronze.stg_store")

In [0]:
display(df.limit(5))

In [0]:
row_count = df.count()
row_count

In [0]:
# DUPLICATE ANALYSIS
print("\nDUPLICATE ANALYSIS")
duplicate_store_ids = df.groupBy("store_id").count().filter(col("count") > 1)
print(f"Duplicate store_ids: {duplicate_store_ids.count()}")

duplicate_names = df.groupBy("store_name").count().filter(col("count") > 1)
print(f"Duplicate store_names: {duplicate_names.count()}")

df = df.dropDuplicates(["store_id"])

In [0]:
#NULL VALUE ANALYSIS
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
print("Null counts by column:")
display(null_counts)

In [0]:
display(df.filter(col("manager_name").isNull()))

In [0]:
# Fill null manager_name using manager_id from other rows
window_spec = Window.partitionBy("manager_id")
df = df.withColumn(
    "manager_name",
    f.first("manager_name", ignorenulls=True).over(window_spec)
)

display(df.orderBy("store_id"))

In [0]:
# Check store_type values
print("\nStore_type distribution:")
df.groupBy("store_type").count().orderBy("count", ascending=False).show()

In [0]:
# Check for whitespace issues

columns_to_fix = ["store_name", "city", "state", "manager_id", "manager_name", "store_type"]

for col_name in columns_to_fix:
    df = df.withColumn(col_name,f.trim(f.regexp_replace( col(col_name)," +"," " )))


In [0]:
# State code validation
print("\nState/Territory codes found:")
df.groupBy("state").count().orderBy("state").show(100, truncate=False)


In [0]:
df = df.withColumnRenamed("_modified", "modified")
df = df.withColumn("modified", f.to_date(col("modified")))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_store")